# Gradient norms: extended report
Расширенный просмотр `mean/max` для всей модели, Gumbel logits и отдельных слоёв. Легенды автоматически используют `reporting.run_label_fields` из resolved config.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "net_complexity").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from net_complexity.studies import (
    gradient_norm_catalog,
    load_study,
    plot_core_statistics,
    plot_gradient_norms,
)

In [ ]:
STUDY_DIR = (REPO_ROOT / "outputs/studies/PUT_STUDY_HERE").resolve()
RUN_NAME = None       # None = все runs; либо имя конкретного run
MAX_LAYERS = 12       # защита от сотен графиков
STATISTICS = ("mean", "max")

summary_df, history_df = load_study(STUDY_DIR)
if RUN_NAME is not None:
    history_df = history_df[history_df["run_name"] == RUN_NAME].copy()
    if history_df.empty:
        raise ValueError(f"Run not found: {RUN_NAME}")

print(f"study: {STUDY_DIR}")
print(f"runs in view: {history_df['run_name'].nunique()}, history: {history_df.shape}")
display(summary_df)

In [ ]:
catalog = gradient_norm_catalog(history_df)
if catalog.empty:
    raise ValueError("В history.csv нет колонок grad_norm_*. Проверь gradient_norm_logging.enabled.")
display(catalog)

In [ ]:
plot_core_statistics(history_df)

groups = set(catalog["parameter_group"])
for parameter_group in ("total", "gumbel_logits_total"):
    if parameter_group not in groups:
        continue
    for statistic in STATISTICS:
        plot_gradient_norms(
            history_df,
            parameter_group=parameter_group,
            statistic=statistic,
            yscale="log",
        )

In [ ]:
layer_groups = sorted(group for group in groups if group.startswith("layer_"))
print(f"layer groups: {len(layer_groups)}; showing: {min(len(layer_groups), MAX_LAYERS)}")

for parameter_group in layer_groups[:MAX_LAYERS]:
    for statistic in STATISTICS:
        available_components = tuple(
            catalog.loc[
                (catalog["parameter_group"] == parameter_group)
                & (catalog["statistic"] == statistic),
                "component",
            ].tolist()
        )
        if available_components:
            plot_gradient_norms(
                history_df,
                components=available_components,
                parameter_group=parameter_group,
                statistic=statistic,
                yscale="log",
            )